In [15]:
# Interpretable Machine Learning for Credit Risk Modeling using SHAP and LIME

# -------------------------
# 0) Install requirements
# -------------------------
!pip install -q xgboost lightgbm shap lime imbalanced-learn scikit-learn joblib

# -------------------------
# 1) Imports
# -------------------------
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os, joblib, textwrap
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer
import random

# -------------------------
# 2) Load dataset
# -------------------------
DATA_PATH = "loan_data_set.csv"   # put the file here in Colab files pane
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Place your dataset at {DATA_PATH} (upload in Colab Files or mount Drive)")

df = pd.read_csv(DATA_PATH)
print("Loaded dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# ---------- Identify target ----------
# This dataset has the classic 'Loan_Status' column with 'Y'/'N' values
TARGET_COL = 'Loan_Status'
if TARGET_COL not in df.columns:
    raise ValueError(f"Expected target column '{TARGET_COL}' in dataset. Found: {df.columns.tolist()}")
df[TARGET_COL] = df[TARGET_COL].map({'Y':1, 'N':0})
if df[TARGET_COL].isnull().any():
    raise ValueError("Null values created when mapping Loan_Status to binary. Check unique values in the target column.")

# -------------------------
# 3) Preprocessing pipeline
# -------------------------
# Drop ID if present
if 'Loan_ID' in df.columns:
    df = df.drop(columns=['Loan_ID'])

num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET_COL]
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical cols:", num_cols)
print("Categorical cols:", cat_cols)
print("Target distribution:\n", df[TARGET_COL].value_counts())

# Pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# -------------------------
# 4) Train/test split & preprocessing fit
# -------------------------
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,
                                                    random_state=42, stratify=y)

# Fit preprocessor on training data
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# derive feature names produced by the preprocessor
ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + cat_feature_names
print("Feature vector length:", len(feature_names))

# -------------------------
# 5) SMOTE to handle imbalance
# -------------------------
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_prep, y_train)
print("After SMOTE:", dict(pd.Series(y_train_bal).value_counts()))

# -------------------------
# 6) Candidate models & hyperparameter tuning
# -------------------------
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
lgb = LGBMClassifier(random_state=42)

xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0]
}
lgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'num_leaves': [15, 31, 63],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0]
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rs_xgb = RandomizedSearchCV(xgb, xgb_param_grid, n_iter=6, scoring='roc_auc', cv=cv, random_state=42, n_jobs=-1, verbose=0)
rs_xgb.fit(X_train_bal, y_train_bal)
best_xgb = rs_xgb.best_estimator_

rs_lgb = RandomizedSearchCV(lgb, lgb_param_grid, n_iter=6, scoring='roc_auc', cv=cv, random_state=42, n_jobs=-1, verbose=0)
rs_lgb.fit(X_train_bal, y_train_bal)
best_lgb = rs_lgb.best_estimator_

print("XGBoost best params:", rs_xgb.best_params_, "CV AUC:", rs_xgb.best_score_)
print("LightGBM best params:", rs_lgb.best_params_, "CV AUC:", rs_lgb.best_score_)

# -------------------------
# 7) Evaluate on test set (AUC, Precision, Recall, F1)
# -------------------------
def evaluate_model(model, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:,1]
    y_pred = (y_proba >= 0.5).astype(int)
    return {
        'auc': roc_auc_score(y_test, y_proba),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'y_proba': y_proba,
        'y_pred': y_pred
    }

res_xgb = evaluate_model(best_xgb, X_test_prep, y_test)
res_lgb = evaluate_model(best_lgb, X_test_prep, y_test)

print("\nXGBoost test metrics:", {k: round(v,4) for k,v in res_xgb.items() if k in ['auc','precision','recall','f1']})
print("LightGBM test metrics:", {k: round(v,4) for k,v in res_lgb.items() if k in ['auc','precision','recall','f1']})

# choose best model by AUC
best_model = best_xgb if res_xgb['auc'] >= res_lgb['auc'] else best_lgb
best_model_name = "XGBoost" if best_model is best_xgb else "LightGBM"
best_res = res_xgb if best_model is best_xgb else res_lgb
print("Selected best model:", best_model_name)

# -------------------------
# 8) Global SHAP explanations
# -------------------------
# Use TreeExplainer if possible for tree models for speed
explainer = shap.Explainer(best_model, feature_names=feature_names)
shap_values = explainer(X_train_prep)  # use entire training set (preprocessed) for global view

# mean absolute shap per feature (global importance)
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_abs_shap})
shap_importance = shap_importance.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

top10 = shap_importance.head(10)
top5 = shap_importance.head(5)

print("\nTop 10 features by mean |SHAP|:")
print(top10.to_string(index=False))

# Compose textual summary of top 5 drivers
global_top5_summary_lines = []
for i, row in top5.iterrows():
    global_top5_summary_lines.append(f"{i+1}. {row['feature']} — mean |SHAP| = {row['mean_abs_shap']:.6f}")
global_top5_summary = "\n".join(global_top5_summary_lines)
print("\nGlobal top-5 drivers (text):\n", global_top5_summary)

# -------------------------
# 9) Local explanations: pick 5 test instances (3 approvals, 2 denials)
# -------------------------
test_df = X_test.reset_index(drop=True).copy()
test_df[TARGET_COL] = y_test

# Ensure sampling possible: if not enough approvals/denials, reduce sample size accordingly
n_approvals = min(3, test_df[test_df[TARGET_COL]==1].shape[0])
n_denials = min(2, test_df[test_df[TARGET_COL]==0].shape[0])

approved_idx = test_df[test_df[TARGET_COL]==1].sample(n=n_approvals, random_state=42).index.tolist()
denied_idx = test_df[test_df[TARGET_COL]==0].sample(n=n_denials, random_state=42).index.tolist()
selected_idx = approved_idx + denied_idx
print("Selected indices (relative to test set):", selected_idx)

selected_rows_original = test_df.loc[selected_idx].drop(columns=[TARGET_COL]).reset_index(drop=True)
selected_vectors = X_test_prep[selected_idx]

# Prepare LIME explainer (needs training matrix)
lime_explainer = LimeTabularExplainer(X_train_bal, feature_names=feature_names, class_names=['No','Yes'], discretize_continuous=True, random_state=42)

local_explanations = []
for i, vec in enumerate(selected_vectors):
    orig_row = selected_rows_original.loc[i].to_dict()
    # SHAP local explanation
    shap_local = explainer(vec.reshape(1, -1))
    # pair features & shap contributions
    contribs = list(zip(feature_names, shap_local.values.flatten()))
    contribs_sorted = sorted(contribs, key=lambda x: abs(x[1]), reverse=True)[:8]
    shap_text_lines = []
    for feat, val in contribs_sorted:
        direction = "increases probability of default" if val > 0 else "decreases probability of default"
        shap_text_lines.append(f"{feat}: {direction} by {abs(val):.6f}")
    shap_text = "\n".join(shap_text_lines)
    # LIME local explanation (text)
    lime_exp = lime_explainer.explain_instance(vec, best_model.predict_proba, num_features=8)
    lime_list = lime_exp.as_list(label=1)
    lime_text_lines = [f"{feat_desc} (weight {weight:.6f})" for feat_desc, weight in lime_list]
    lime_text = "\n".join(lime_text_lines)

    pred_proba = best_model.predict_proba(vec.reshape(1,-1))[0,1]
    pred_label = int(best_model.predict(vec.reshape(1,-1))[0])
    local_explanations.append({
        'index_in_test_set': selected_idx[i],
        'original_values': orig_row,
        'pred_proba': float(pred_proba),
        'pred_label': pred_label,
        'shap_text': shap_text,
        'lime_text': lime_text
    })

# Print local explanations
for le in local_explanations:
    print("\n--- Instance (test idx {}) ---".format(le['index_in_test_set']))
    print("Original values:")
    for k,v in le['original_values'].items():
        print(f"  {k}: {v}")
    print(f"Model predicted probability of default: {le['pred_proba']:.6f}, predicted label: {le['pred_label']}")
    print("\nSHAP local explanation (top contributors):")
    print(le['shap_text'])
    print("\nLIME local explanation (approximate local linear contributions):")
    print(le['lime_text'])

# -------------------------
# 10) Critical comparison (text)
# -------------------------
comparison_text = textwrap.dedent(f"""
Critical comparison (global SHAP vs local SHAP & LIME):

Global SHAP (top drivers):
{global_top5_summary}

Observations on consistency/differences:
- Global SHAP identifies features that, on average across the training population, have the largest impact on predictions.
- Local SHAP shows how those (or other) features pushed a specific instance toward approval/denial. For many instances the top global features are present among top local contributors, but the sign (increase/decrease) depends on the instance's values.
- LIME returns a locally linear approximation and sometimes selects different top features than SHAP because it fits a simple surrogate model in a neighborhood of the instance. When the model is highly nonlinear or when feature interactions matter, LIME and SHAP can disagree.
- Practical takeaway: Use global SHAP to audit and design policies (e.g., tighten thresholds on top drivers). Use local SHAP and LIME for case-level explanations to customers/regulators, but reconcile differences by inspecting feature interactions and similar cases.

Short recommendation for underwriting policy:
- Base portfolio-level policy on consistent global drivers (top features above).
- For exceptions/appeals, include local SHAP explanations in case files and trigger a manual review when local explanations conflict with policy.
""")
print(comparison_text)

# -------------------------
# 11) Save a structured text report (Deliverable 2)
# -------------------------
report_lines = []
report_lines.append("# Interpretable Credit Risk Model — Structured Report\n")
report_lines.append("## 1. Hyperparameter choices (tuned)\n")
report_lines.append("### XGBoost best params:\n" + str(rs_xgb.best_params_) + "\n")
report_lines.append("### LightGBM best params:\n" + str(rs_lgb.best_params_) + "\n")

report_lines.append("## 2. Final model performance on test set\n")
report_lines.append(f"Selected model: {best_model_name}\n")
report_lines.append(f"AUC: {best_res['auc']:.6f}\nPrecision: {best_res['precision']:.6f}\nRecall: {best_res['recall']:.6f}\nF1: {best_res['f1']:.6f}\n")

report_lines.append("## 3. Global feature importance (Top-5 drivers)\n")
report_lines.append(global_top5_summary + "\n")

report_lines.append("## 4. Local explanations (5 cases):\n")
for j, le in enumerate(local_explanations, start=1):
    report_lines.append(f"### Case {j} (test index {le['index_in_test_set']})\n")
    report_lines.append("Original feature values:\n")
    for k,v in le['original_values'].items():
        report_lines.append(f"- {k}: {v}\n")
    report_lines.append(f"Predicted probability of default: {le['pred_proba']:.6f}\nPredicted label: {le['pred_label']}\n")
    report_lines.append("SHAP local (top contributors):\n")
    report_lines.append(le['shap_text'] + "\n")
    report_lines.append("LIME local (top contributions):\n")
    report_lines.append(le['lime_text'] + "\n")

report_lines.append("## 5. Critical comparison and policy implications\n")
report_lines.append(comparison_text + "\n")

report_text = "\n".join(report_lines)
REPORT_PATH = "/content/interpretability_report.md"
with open(REPORT_PATH, "w") as f:
    f.write(report_text)

print("Saved structured report to:", REPORT_PATH)

# -------------------------
# 12) Save artifacts (model & preprocessor)
# -------------------------
artifacts = {'preprocessor': preprocessor, 'best_model': best_model, 'feature_names': feature_names}
joblib.dump(artifacts, "/content/credit_model_artifacts.pkl")
print("Saved model artifacts to /content/credit_model_artifacts.pkl")

# -------------------------
# End of script
# -------------------------
print("\nAll done. Open the markdown report at /content/interpretability_report.md for the structured Deliverable 2.")


Loaded dataset shape: (614, 13)
Columns: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']
Numerical cols: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
Categorical cols: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
Target distribution:
 Loan_Status
1    422
0    192
Name: count, dtype: int64
Feature vector length: 20
After SMOTE: {1: np.int64(337), 0: np.int64(337)}
[LightGBM] [Info] Number of positive: 337, number of negative: 337
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 736
[LightGBM] [Info] Number of data points in the train set: 674, numbe